# 🔌 GIADA supplemento Task 15c — ponte d'interfaccia verso il teacher
La MLP della Task 15b richiede quattro correnti locali pianificate che il teacher multicompartimentale non fornisce in forma equivalente. **Non** le sostituiamo con zeri o con valori ricavati dal futuro. Confrontiamo invece candidati gate già congelati, capaci di ricevere V a ogni sottopasso, sui soli percorsi `validation` del teacher. Gli split di test rimangono sigillati per la Task 16. Nessun retraining o sostituzione di corrente/membrana.


In [ ]:
from pathlib import Path
import base64,hashlib,json,os,shutil,subprocess,sys,zipfile
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task15c');REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
assert not WORK.exists(),f'Workspace già presente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_embedded_interface_bridge.py').is_file(),'Revisione senza Task 15c.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula
from src.giada_teacher.roadmap_embedded_interface_bridge import EmbeddedInterfaceConfig,verified_task15b_result,verified_task5_any,run_interface_bridge
print({'revision':REVISION})


## 📁 Tre input verificati
Aggiungi: (1) dataset `hayflow-targeted-transition-dataset-v1-1-base` con `transition_dataset.h5` (6 GiB, o `archive.zip`), (2) `giada_primitive_scaling_laws.zip` della Task 5, (3) `giada_roadmap_task15b_gate_bottleneck_confirmation.zip`. Il notebook verifica SHA di report/checkpoint e HDF5; non carica altri dataset storici. Se i nomi del mount differiscono, imposta `GIADA_TARGETED_DATASET`, `GIADA_TASK5_ARTIFACT` e `GIADA_TASK15B_ARTIFACT`.


In [ ]:
INPUT=Path('/kaggle/input')
override=os.environ.get('GIADA_TASK15B_ARTIFACT');candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += [p for p in INPUT.rglob('*.zip') if 'task15b' in str(p).lower() or 'gate-bottleneck' in str(p).lower()]
 candidates += [p.parent for p in INPUT.rglob('final_report.json') if 'task15b' in str(p).lower()]
TASK15B=None;verified15b=None
for path in dict.fromkeys(candidates):
 if not path.exists():continue
 try:verified15b=verified_task15b_result(path);TASK15B=path.resolve();break
 except (ValueError,KeyError,FileNotFoundError,OSError):continue
assert TASK15B is not None,'Artefatto Task 15b esatto non trovato.'
override=os.environ.get('GIADA_TASK5_ARTIFACT');candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += list(INPUT.rglob('giada_primitive_scaling_laws.zip'))
 candidates += [p.parent for p in INPUT.rglob('frozen_scaling_checkpoints.pt')]
 candidates += [p for p in INPUT.rglob('archive.zip') if 'primitive_scaling' in str(p).lower() or 'primitive-scaling' in str(p).lower()]
TASK5=None
for attempt,path in enumerate(dict.fromkeys(candidates)):
 if not path.exists():continue
 try:verified_task5_any(path,WORK/f'.task5_discovery_{attempt}');TASK5=path.resolve();break
 except (ValueError,RuntimeError,FileNotFoundError,OSError,zipfile.BadZipFile):pass
assert TASK5 is not None,'Artefatto Task 5 esatto non trovato: aggiungi lo ZIP originale o la cartella estratta.'
override=os.environ.get('GIADA_TARGETED_DATASET');candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():
 candidates += [p.parent for p in INPUT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
 candidates += [p for p in INPUT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE=next((p.resolve() for p in dict.fromkeys(candidates) if p.exists()),None)
assert BASE is not None,'Dataset targeted v1.1 base non trovato.'
display({'task15b':str(TASK15B),'task15b_verified':verified15b,'task5':str(TASK5),'base':str(BASE)})


In [ ]:
def materialize_base(source):
 source=Path(source)
 if source.is_dir():
  assert (source/'dataset_manifest.json').is_file() and (source/'state_schema.json').is_file(),'Base incompleta: manifest o schema mancanti.'
  return source
 destination=WORK/'extracted_base';assert not destination.exists(),f'Estrazione già presente: {destination}'
 destination.mkdir(parents=True)
 with zipfile.ZipFile(source) as archive:
  required=('dataset_manifest.json','state_schema.json','transition_dataset.h5')
  members={name:[m for m in archive.infolist() if m.filename.endswith('/'+name)] for name in required}
  assert all(len(items)==1 for items in members.values()),'Archivio base ambiguo o incompleto.'
  assert len({str(Path(items[0].filename).parent) for items in members.values()})==1,'File base da directory diverse.'
  for name in required:
   member=members[name][0];print({'extracting':name,'GiB':round(member.file_size/2**30,3)},flush=True)
   with archive.open(member) as reader,(destination/name).open('xb') as writer:
    copied=0;notice=.1
    while True:
     block=reader.read(16*1024*1024)
     if not block:break
     writer.write(block);copied+=len(block)
     if name.endswith('.h5') and copied/member.file_size>=notice:
      print(f'[GIADA 15c] estrazione HDF5 {round(100*copied/member.file_size)}%',flush=True);notice+=.1
 return destination
BASE_ROOT=materialize_base(BASE)
print({'base_root':str(BASE_ROOT),'hash_verification':'nella cella seguente'})


## 🔬 Confronto congelato sullo split validation
L'hash completo dei 6 GiB è obbligatorio e stampa progressi. Campioniamo fino a 16 percorsi per sito/regime fra soma, hot-zone e due tuft. Confrontiamo tre viste di V: campione sinistro disponibile al sottopasso, midpoint teacher-forced, campione destro oracle. Formula, LUT-513 e tre seed `physical_tau` sono confrontati senza training. Il floor formula–teacher determina se i numeri candidato–teacher siano interpretabili. **Il checkpoint 15b non viene valutato con correnti locali inventate.**


In [ ]:
import torch
assert torch.cuda.is_available(),'La Task 15c richiede GPU CUDA per i candidati Task 5 congelati.'
config=EmbeddedInterfaceConfig();config.validate()
formula=ExtractedGateFormula.from_mod(TEACHER/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task15c_embedded_interface_bridge')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Avvia una sessione pulita.'
def progress(*items):
 if len(items)==2:print(f'[GIADA 15c][SHA-256 {items[1]}] {items[0]}%',flush=True)
 else:print(f'[GIADA 15c] scan {items[0]}/{items[1]} selezionati={items[2]} elapsed={items[3]:.1f} min',flush=True)
report=run_interface_bridge(formula,BASE_ROOT,TASK5,TASK15B,OUTPUT,config,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'support':report['support'],'floor_calibrated':report['midpoint_formula_floor_calibrated'],'scores':report['development_macro_gate_scores'],'selected_for_task16':report['teacher_forced_candidate_for_task16'],'test_opened':report['teacher_test_splits_opened'],'task15b_fake_input_evaluated':report['task15b_checkpoint_evaluated_in_teacher']})
assert report['valid'] and not report['teacher_test_splits_opened'] and not report['task15b_checkpoint_evaluated_in_teacher']


## 📦 Scarica il report
Lo ZIP contiene soltanto report e indici di provenienza, non il dataset da 6 GiB. Usa il metodo Blob/base64 concordato. Scaricalo anche se il floor non è calibrato: in quel caso il NO-GO è un risultato da analizzare, non un motivo per cambiare la soglia.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task15c_embedded_interface_bridge','zip',OUTPUT.parent,OUTPUT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
